# Counterfit Drone Detection Assessment - Starter

Complete this notebook to configure a binary drone detection image classifier as a Microsoft Counterfit target, run at least three attacks, and summarize operational impact.

The classroom model uses CIFAR-10 airplane images as a lightweight proxy for drone imagery.

In [ ]:
from pathlib import Path
import json
import sys
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT / 'src'))
sys.path.append(str(ROOT))

from drone_counterfit_utils import (
    evaluate_detection_metrics,
    load_npz_dataset,
    require_counterfit,
    run_counterfit_attack_plan,
    write_results_csv,
)
from targets.drone_detection_target import DroneDetectionTarget

attack_plan = json.loads((ROOT / 'configs' / 'counterfit_attack_plan.json').read_text())
target_config = json.loads((ROOT / 'configs' / 'counterfit_target_config.json').read_text())
target_config

## 1. Complete the target scaffold

Open `targets/drone_detection_target.py` and implement the `predict` method. Counterfit expects a batch of model probabilities returned as a Python list.

In [ ]:
counterfit, Counterfit = require_counterfit()

target = DroneDetectionTarget()
target.load()

val_images, val_labels = load_npz_dataset(ROOT / target.data_path)
baseline = evaluate_detection_metrics(target.model, val_images, val_labels, device=target.device)

print(f"Validation images: {len(val_labels)}")
print(f"Baseline accuracy: {baseline['accuracy']:.3f}")
print(f"Drone recall: {baseline['drone_recall']:.3f}")
print(f"Baseline false-negative rate: {baseline['false_negative_rate']:.3f}")
print(f"Mean confidence: {baseline['mean_confidence']:.3f}")

## 2. Review and tune the attack plan

The starter config already includes three attacks. You may adjust attack parameters, but keep at least three attack modules in the plan.

In [ ]:
# TODO: Inspect the configured attacks and confirm the sample indexes target drone-proxy examples.
attack_plan

## 3. Execute Counterfit attacks and save results

In [ ]:
# TODO: Run the attack plan after the target predict method is complete.
attack_rows = run_counterfit_attack_plan(target, attack_plan)

results_path = write_results_csv(attack_rows, ROOT / 'results' / 'starter_counterfit_drone_results.csv')
print(results_path)
attack_rows

## 4. Visualize prediction changes

In [ ]:
# TODO: Generate a comparison figure using the clean validation images and final labels.
sample_indexes = attack_plan['sample_index']
if not isinstance(sample_indexes, list):
    sample_indexes = [sample_indexes]
fig, axes = plt.subplots(len(attack_rows), len(sample_indexes), figsize=(6, 6))
axes = np.array(axes).reshape(len(attack_rows), len(sample_indexes))
for row_idx, row in enumerate(attack_rows):
    final_labels = row['final_labels'].split('|') if row['final_labels'] else []
    for col_idx, sample_idx in enumerate(sample_indexes):
        ax = axes[row_idx][col_idx]
        ax.imshow(val_images[sample_idx].transpose(1, 2, 0))
        predicted = final_labels[col_idx] if col_idx < len(final_labels) else 'unknown'
        ax.set_title(f"{row['attack']}\nfinal={predicted}", fontsize=8)
        ax.axis('off')

plt.tight_layout()
figure_path = ROOT / 'results' / 'starter_prediction_changes.png'
figure_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(figure_path, dpi=160)
figure_path

## 5. Report operational impact

Use the output CSV and Counterfit logs to complete `docs/assessment_report_template.md`. Focus on attack success rate, adversarial accuracy, confidence degradation, and drone false-negative rate.